In [10]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [11]:
web_search_schema = {
    "type": "web_search_20260209",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

In [12]:
messages = []
add_user_message(
    messages,
    """
    What's the bets exercise for gaining leg muscle?
    Use your tools to search the web for the best exercise for gaining leg muscle.
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_01RRKtxpFMw7amFrt86uiuka', container=Container(id='container_0147AZmPBzahGEgBxjXQvdEH', expires_at=datetime.datetime(2026, 6, 15, 1, 1, 29, 836861, tzinfo=TzInfo(0))), content=[ServerToolUseBlock(id='srvtoolu_019RrFX8kXHpg9KXFqg6JqGg', caller=None, input={'code': '\nimport json\n\n# Search for the best exercise for gaining leg muscle\nresult = await web_search({"query": "best exercise for gaining leg muscle"})\nparsed = json.loads(result)\n\n# Print a summary of the search results\nprint(f"Found {len(parsed)} search results\\n")\n\nfor i, result in enumerate(parsed):\n    if result.get(\'type\') == \'error\':\n        print(f"Error: {result.get(\'error_code\')}")\n    else:\n        print(f"Result {i}: {result.get(\'title\')}")\n        print(f"URL: {result.get(\'url\')}")\n        # Print just first 200 chars of content to avoid excessive output\n        content = result.get(\'content\', \'\')\n        print(f"Preview: {content[:200]}...\\n")\n'}, name='code_execution'